# Stock Market Historical Data Analysis and Visualization

**Course:** EDAV (22ADC32N) - Course-End Project | **Dept.:** Information Technology, CBIT | **Sem:** III (2026-27)

**Objective:** Analyze historical stock price data to explore return, risk and trading-volume patterns using NumPy, Pandas and visualization tools, and develop insights into the factors that influence daily stock performance.

**Workflow (as required):** Data Cleaning -> Analysis -> Visualization -> Conclusion

| Titanic sample | This project (Stock Market) |
|---|---|
| Survived / Not survived | Up day / Down day (sign of daily return) |
| Age | Daily return %, Close price, Volume |
| Pclass, Sex | Year, Month, Weekday, Volume regime |
| Embarked port | Ticker (company / stock) |

**Notebook order:** Setup -> Data inspection -> **Q3 (cleaning, run first)** -> Q1 -> Q2 -> Q4 -> Q5 -> Final conclusion.

## 0. Environment Setup

```
pip install numpy pandas matplotlib seaborn yfinance
```
`yfinance` is only needed once, to download the dataset. If you already have a CSV (e.g. from Kaggle), skip the download cell and place it next to this notebook as `stock_data.csv` with columns: `Date, Ticker, Open, High, Low, Close, Volume`.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
pd.set_option('display.max_columns', 30)
np.random.seed(42)   # reproducibility

In [ ]:
# ---- Dataset download (runs only if stock_data.csv does not exist) ----
CSV_PATH = 'stock_data.csv'
TICKERS  = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS', 'ICICIBANK.NS']   # change if you like
START, END = '2021-01-01', '2025-12-31'

if not os.path.exists(CSV_PATH):
    import yfinance as yf
    frames = []
    for t in TICKERS:
        d = yf.download(t, start=START, end=END, auto_adjust=True, progress=False)
        if isinstance(d.columns, pd.MultiIndex):          # newer yfinance returns MultiIndex columns
            d.columns = d.columns.get_level_values(0)
        d = d.reset_index()[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
        d.insert(1, 'Ticker', t.replace('.NS', ''))
        frames.append(d)
    pd.concat(frames, ignore_index=True).to_csv(CSV_PATH, index=False)
    print('Downloaded and saved ->', CSV_PATH)
else:
    print(CSV_PATH, 'already exists - using it.')

## 1. Dataset Inspection

**Concept:** Before any analysis we inspect shape, data types, missing values, duplicates and basic statistics to find problems that must be fixed.

In [ ]:
raw = pd.read_csv(CSV_PATH)
print('Shape:', raw.shape)
print('Tickers:', raw['Ticker'].unique().tolist())
raw.info()
display(raw.head())

In [ ]:
print('Missing values per column:')
display(raw.isnull().sum())
print('Fully duplicated rows        :', raw.duplicated().sum())
print('Duplicate (Date, Ticker) rows:', raw.duplicated(subset=['Date', 'Ticker']).sum())
display(raw.describe())

**Observation:** *(write in your own words after running - e.g. number of rows, tickers, date range, which columns have missing values/duplicates, whether Date is stored as text.)*

## Q3. Handling Missing Values and Anomalies (Imputation)  `[CO3, BL3]`

**Concept:** Stock data is a *time series*, so we cannot blindly use the median like Age in Titanic - a median price from 2021 would be meaningless in 2025. For **prices** we use *forward-fill then back-fill within each ticker* (a missing day takes the last known price). For **Volume** (not continuous in time) we use the *median of that ticker*. We also fix types, remove duplicates, treat impossible values as missing, and engineer features used in later questions.

In [ ]:
df = raw.copy()
price_cols = ['Open', 'High', 'Low', 'Close']
num_cols   = price_cols + ['Volume']

# 1) Fix data types
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# 2) Drop rows without a valid Date/Ticker and remove duplicates
print('Rows with invalid Date/Ticker:', df[['Date', 'Ticker']].isna().any(axis=1).sum())
df = df.dropna(subset=['Date', 'Ticker'])
n_dup = df.duplicated(subset=['Date', 'Ticker']).sum()
df = df.drop_duplicates(subset=['Date', 'Ticker'], keep='first')
print('Duplicates removed:', n_dup)

df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# 3) Anomalies -> treat as missing, then impute
bad_price  = (df[price_cols] <= 0).any(axis=1)
bad_range  = df['High'] < df['Low']
bad_volume = df['Volume'] < 0
print('Non-positive prices :', bad_price.sum())
print('High < Low rows     :', bad_range.sum())
print('Negative volume rows:', bad_volume.sum())
df.loc[bad_price | bad_range, price_cols] = np.nan
df.loc[bad_volume, 'Volume'] = np.nan

print('\nMissing values BEFORE imputation:')
display(df[num_cols].isnull().sum())

In [ ]:
# 4) Imputation (done separately for every ticker)
df[price_cols] = df.groupby('Ticker')[price_cols].transform(lambda s: s.ffill().bfill())
df['Volume']   = df.groupby('Ticker')['Volume'].transform(lambda s: s.fillna(s.median()))

# 5) Verify - no missing values may remain
assert df[num_cols + ['Date', 'Ticker']].isnull().sum().sum() == 0
print('Missing values AFTER imputation:', int(df[num_cols].isnull().sum().sum()))

In [ ]:
# 6) Feature engineering
df['Daily_Return'] = df.groupby('Ticker')['Close'].pct_change() * 100          # in %
df['Direction']    = np.select([df['Daily_Return'] > 0, df['Daily_Return'] < 0],
                               ['Up', 'Down'], default='Flat')
df['Year']    = df['Date'].dt.year
df['Month']   = df['Date'].dt.month
df['Weekday'] = df['Date'].dt.day_name()
df['MA20']    = df.groupby('Ticker')['Close'].transform(lambda s: s.rolling(20).mean())
df['MA50']    = df.groupby('Ticker')['Close'].transform(lambda s: s.rolling(50).mean())

# first row of every ticker has no previous day -> no return; keep it out of return-based analysis
ret = df.dropna(subset=['Daily_Return']).copy()
print('Rows in clean dataset:', len(df), '| rows used for return analysis:', len(ret))
display(df.head())

In [ ]:
# 7) Outlier check on daily returns (flagged, NOT removed - real crashes/rallies are genuine data)
ret['Z'] = ret.groupby('Ticker')['Daily_Return'].transform(lambda s: (s - s.mean()) / s.std())
outliers = ret[ret['Z'].abs() > 4]
print('Days with |z| > 4:', len(outliers))
display(ret.loc[ret['Daily_Return'].abs().nlargest(8).index, ['Date', 'Ticker', 'Close', 'Daily_Return', 'Volume']])

fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=ret, x='Ticker', y='Daily_Return', ax=ax)
ax.set_title('Daily Return Distribution by Ticker (outlier check)')
ax.set_xlabel('Ticker'); ax.set_ylabel('Daily Return (%)')
plt.show()

FOCUS = df['Ticker'].unique()[0]      # stock used for single-stock analysis (change if you want)
print('Focus stock:', FOCUS)

**Conclusion / Insight (Q3):** *(your words)* - mention how many values were missing/invalid, why forward-fill was chosen for prices instead of the median, how many outliers were found, and why they were kept (large moves are real market events, e.g. results days or market-wide shocks).

## Q1. NumPy Statistics - Average, Minimum and Maximum by Group  `[CO1, BL3]`

**Concept:** Using only NumPy array operations (boolean masks, `np.mean`, `np.min`, `np.max`) we compute statistics for the focus stock, grouped by (a) **market direction** (Up vs Down days - the analogue of Survived vs Not survived), (b) **volume regime** (high vs low volume days) and (c) **year**.

In [ ]:
f = ret[ret['Ticker'] == FOCUS]
returns   = f['Daily_Return'].to_numpy()
volume    = f['Volume'].to_numpy()
close     = f['Close'].to_numpy()
direction = f['Direction'].to_numpy()
years     = f['Year'].to_numpy()

# (a) By direction (Up / Down / Flat)
rows = []
for d in ['Up', 'Down', 'Flat']:
    m = direction == d
    if m.sum() == 0:
        continue
    rows.append({'Direction': d, 'Days': int(m.sum()),
                 'Avg Return %': np.mean(returns[m]), 'Min Return %': np.min(returns[m]),
                 'Max Return %': np.max(returns[m]), 'Avg Volume': np.mean(volume[m])})
print(f'(a) Return and volume by direction - {FOCUS}')
display(pd.DataFrame(rows).set_index('Direction'))

# (b) By volume regime
high = volume > np.median(volume)
rows = []
for name, m in [('High volume', high), ('Low volume', ~high)]:
    rows.append({'Regime': name, 'Days': int(m.sum()),
                 'Avg |Return| %': np.mean(np.abs(returns[m])), 'Min Return %': np.min(returns[m]),
                 'Max Return %': np.max(returns[m]), 'Std Return %': np.std(returns[m])})
print(f'\n(b) Return behaviour by volume regime - {FOCUS}')
display(pd.DataFrame(rows).set_index('Regime'))
print('Correlation between Volume and |Return|:', round(np.corrcoef(volume, np.abs(returns))[0, 1], 3))

# (c) Close price by year
rows = []
for y in np.unique(years):
    m = years == y
    rows.append({'Year': int(y), 'Avg Close': np.mean(close[m]), 'Min Close': np.min(close[m]),
                 'Max Close': np.max(close[m]), 'Avg Daily Return %': np.mean(returns[m])})
print(f'\n(c) Close price statistics by year - {FOCUS}')
display(pd.DataFrame(rows).set_index('Year'))

**Conclusion / Insight (Q1):** *(your words)* - e.g. compare size of average up-day vs down-day moves, whether high-volume days are more volatile (see correlation), and which year had the highest average price / best average return.

## Q2. Pandas `groupby` - How Time Period and Stock Influence Returns  `[CO2, BL4]`

**Concept:** Like analysing survival by class and gender, we use `groupby` + aggregation to see how **year, month and weekday** influence average return, volatility (standard deviation) and the percentage of up days ("win rate"), and then compare the stocks overall.

In [ ]:
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

def summarize(g):
    return g.agg(Avg_Return=('Daily_Return', 'mean'),
                 Volatility=('Daily_Return', 'std'),
                 Up_Day_Pct=('Daily_Return', lambda s: (s > 0).mean() * 100),
                 Avg_Volume=('Volume', 'mean'))

yearly  = summarize(ret.groupby(['Ticker', 'Year']))
weekday = summarize(ret.groupby('Weekday')).reindex(order).dropna(how='all')
monthly = summarize(ret.groupby('Month'))

print('Year-wise performance by ticker'); display(yearly)
print('\nWeekday effect (all tickers)');   display(weekday)
print('\nMonth-wise (all tickers)');       display(monthly)

In [ ]:
# Overall stock comparison
first_last = df.groupby('Ticker')['Close'].agg(First='first', Last='last')
overall = pd.DataFrame({
    'Total_Return_%': (first_last['Last'] / first_last['First'] - 1) * 100,
    'Avg_Daily_Return_%': ret.groupby('Ticker')['Daily_Return'].mean(),
    'Annual_Volatility_%': ret.groupby('Ticker')['Daily_Return'].std() * np.sqrt(252),
    'Up_Day_%': ret.groupby('Ticker')['Daily_Return'].apply(lambda s: (s > 0).mean() * 100),
})
overall['Return_per_Risk'] = overall['Avg_Daily_Return_%'] * 252 / overall['Annual_Volatility_%']
display(overall.sort_values('Total_Return_%', ascending=False))

**Conclusion / Insight (Q2):** *(your words)* - which year was best/worst, whether any weekday or month shows consistently higher returns (and whether the difference is large or just noise), which stock gave the best return and which had the best return for the risk taken.

## Q4. Group by Ticker and Direction - Descriptive Statistics  `[CO4, BL4]`

**Concept:** Equivalent of grouping by `['Embarked','Survived']`: we use `groupby(['Ticker','Direction'])` with `count` and `mean` aggregations to compare how often each stock moves up or down and how large the moves are.

In [ ]:
grp = (ret[ret['Direction'].isin(['Up', 'Down'])]
       .groupby(['Ticker', 'Direction'])
       .agg(Days=('Daily_Return', 'count'),
            Avg_Return=('Daily_Return', 'mean'),
            Avg_Volume=('Volume', 'mean'),
            Largest_Move=('Daily_Return', lambda s: s.abs().max())))
display(grp)

counts = ret.pivot_table(index='Ticker', columns='Direction', values='Daily_Return', aggfunc='count').fillna(0)
counts['Up_Probability_%'] = counts['Up'] / (counts['Up'] + counts['Down']) * 100
print('\nUp/Down day counts and probability of an up day')
display(counts)

print('\nFull descriptive statistics of daily return (%) by ticker')
display(ret.groupby('Ticker')['Daily_Return'].describe())

**Conclusion / Insight (Q4):** *(your words)* - which ticker has the highest/lowest probability of an up day, whether average gains are bigger than average losses for each stock, and which stock is riskiest (largest move, widest spread).

## Q5. Bar Plots, Line Plots and Heatmaps  `[CO5, BL5]`

**Concept:** Visualizations make the patterns from Q1-Q4 easy to interpret: line plots for price growth, bar plots for yearly and weekday performance, and heatmaps for monthly returns, up/down counts and correlation between stocks.

In [ ]:
# 5.1 Price growth (all prices rebased to 100 on the first day)
close_pivot = df.pivot(index='Date', columns='Ticker', values='Close')
rebased = close_pivot.apply(lambda s: s / s.dropna().iloc[0] * 100)

fig, ax = plt.subplots(figsize=(11, 5))
rebased.plot(ax=ax, linewidth=1.5)
ax.set_title('Growth of 100 Invested at Start of Period')
ax.set_xlabel('Date'); ax.set_ylabel('Value (rebased to 100)')
ax.legend(title='Ticker')
plt.show()

In [ ]:
# 5.2 Bar plot - average daily return by year and ticker
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=yearly.reset_index(), x='Year', y='Avg_Return', hue='Ticker', ax=ax)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Average Daily Return by Year and Ticker')
ax.set_xlabel('Year'); ax.set_ylabel('Average Daily Return (%)')
ax.legend(title='Ticker', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.show()

# 5.3 Bar plot - probability of an up day by weekday
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(x=weekday.index, y=weekday['Up_Day_Pct'], ax=ax, color='steelblue')
ax.axhline(50, color='red', linestyle='--', label='50% line')
ax.set_title('Percentage of Up Days by Weekday (all tickers)')
ax.set_xlabel('Weekday'); ax.set_ylabel('Up Days (%)')
ax.legend()
plt.show()

In [ ]:
# 5.4 Heatmap - monthly average return (focus stock)
pt = ret[ret['Ticker'] == FOCUS].pivot_table(index='Year', columns='Month', values='Daily_Return', aggfunc='mean')
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(pt, annot=True, fmt='.2f', cmap='RdYlGn', center=0, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Avg daily return (%)'})
ax.set_title(f'{FOCUS}: Average Daily Return (%) by Year and Month')
ax.set_xlabel('Month'); ax.set_ylabel('Year')
plt.show()

# 5.5 Heatmap - number of up / down days per ticker (analogue of survival counts by port)
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(counts[['Up', 'Down']], annot=True, fmt='.0f', cmap='Blues', linewidths=0.5, ax=ax)
ax.set_title('Number of Up and Down Days by Ticker')
ax.set_xlabel('Direction'); ax.set_ylabel('Ticker')
plt.show()

# 5.6 Heatmap - correlation of daily returns between stocks
corr = ret.pivot(index='Date', columns='Ticker', values='Daily_Return').corr()
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Correlation of Daily Returns')
plt.show()

In [ ]:
# 5.7 Price with moving averages and volume (focus stock)
f_all = df[df['Ticker'] == FOCUS]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(f_all['Date'], f_all['Close'], label='Close', linewidth=1.2)
ax1.plot(f_all['Date'], f_all['MA20'], label='20-day MA', linewidth=1)
ax1.plot(f_all['Date'], f_all['MA50'], label='50-day MA', linewidth=1)
ax1.set_title(f'{FOCUS}: Close Price with Moving Averages and Volume')
ax1.set_ylabel('Price'); ax1.legend()
ax2.bar(f_all['Date'], f_all['Volume'], color='gray', width=1.0)
ax2.set_xlabel('Date'); ax2.set_ylabel('Volume')
plt.tight_layout()
plt.show()

**Conclusion / Insight (Q5):** *(your words)* - which stock grew most in the line plot, which years were strongest in the bar plot, whether any weekday is clearly better than 50%, which months look consistently green/red in the heatmap, and how strongly the stocks move together (diversification point from the correlation heatmap).

## Auto-generated Key Facts (use as a cross-check for your conclusions)

In [ ]:
best  = overall['Total_Return_%'].idxmax()
worst = overall['Total_Return_%'].idxmin()
risky = overall['Annual_Volatility_%'].idxmax()
up_t  = counts['Up_Probability_%'].idxmax()
best_wd = weekday['Avg_Return'].idxmax()
worst_wd = weekday['Avg_Return'].idxmin()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()

print(f'Date range                 : {df.Date.min().date()} to {df.Date.max().date()}')
print(f'Best / worst total return  : {best} ({overall.loc[best, "Total_Return_%"]:.1f}%) / {worst} ({overall.loc[worst, "Total_Return_%"]:.1f}%)')
print(f'Most volatile stock        : {risky} ({overall.loc[risky, "Annual_Volatility_%"]:.1f}% annualised)')
print(f'Highest up-day probability : {up_t} ({counts.loc[up_t, "Up_Probability_%"]:.1f}%)')
print(f'Best / worst weekday       : {best_wd} / {worst_wd}')
print(f'Most correlated pair       : {pairs.idxmax()} ({pairs.max():.2f}); least: {pairs.idxmin()} ({pairs.min():.2f})')

## Final Conclusion

*(Write 5-8 lines in your own words summarising: data quality issues and how you fixed them, return and risk differences between stocks, time-based patterns (year/weekday/month), role of volume, correlation/diversification, and limitations - past performance does not predict future returns, short data window, prices only (no news/fundamentals).)*

**Limitations & Future Work:** add more stocks/sectors, include a benchmark index (NIFTY 50), try risk-adjusted metrics (Sharpe ratio), and build simple forecasting models.